# Sistema de Detección de Intrusiones basado en Inteligencia Artificial
## Fortalecimiento de la Ciberseguridad en Redes TCP/IP

**Autor:** Nicolás Martínez Pineda
**Asignatura:** Fundamentos de Redes y Comunicación — Universidad Distrital Francisco José de Caldas
**Docente:** Octavio José Salcedo Parra

Este cuaderno documenta el desarrollo experimental del proyecto conforme a la metodología basada en CRISP-DM
(Chapman et al., 2000): comprensión del problema, preparación de datos, modelado, evaluación y despliegue,
integrando el módulo de identificación de huella digital de dispositivos (IEEE 802.3) con los modelos
de clasificación **Random Forest** (Breiman, 2001) y **XGBoost** (Chen & Guestrin, 2016).

**Estructura del cuaderno:**
1. Configuración del entorno
2. Fase 1 — Comprensión del problema (variables y vectores de ataque por capa)
3. Fase 2 — Captura y extracción de huella digital (parámetros IEEE 802.3)
4. Fase 3 — Preparación del conjunto de datos de tráfico (legítimo / malicioso)
5. Fase 4 — Modelado: Random Forest vs. XGBoost
6. Fase 5 — Evaluación comparativa (precisión, sensibilidad, F1-score, tasa de falsos positivos)
7. Fase 6 — Simulación de ataque a la huella digital y evaluación de la defensa
8. Fase 7 — Diagnóstico, mitigación y pronóstico


In [ ]:
# 1. Configuración del entorno e importación de la biblioteca src/
# Ejecutar solo si hace falta instalar dependencias:
# !pip install -e .. --quiet

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Agregar la raíz del proyecto al path para importar de src/
RAIZ_PROYECTO = Path("..").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.data import RANDOM_STATE
np.random.seed(RANDOM_STATE)
print(f"Entorno configurado. Raíz del proyecto: {RAIZ_PROYECTO}")


## Fase 1 — Comprensión del problema

Vectores de ataque considerados por capa (TCP/IP e ISO/OSI), según la caracterización documental del proyecto:

| Capa | Vector de ataque | Relevancia para la huella digital |
|---|---|---|
| Enlace | ARP spoofing | Altera la asociación IP–MAC, afecta directamente la huella IEEE 802.3 |
| Red | Suplantación de IP, DDoS | Afecta el volumen y origen del tráfico |
| Transporte | SYN flood | Afecta patrones de establecimiento de conexión |
| Aplicación | DNS spoofing, phishing | Afecta el comportamiento de la capa superior, no la huella física |

Estas categorías guían las variables que se extraen en la Fase 2 y las que se usan como entrada de los modelos en la Fase 4.


In [ ]:
# 2. Extracción de huella digital de dispositivos (IEEE 802.3)
# Importado directamente desde src/fingerprint.py (fuente única de verdad)

from src.fingerprint import ExtractorHuellaDigital, oui_a_entero, HuellaDigital
from src.feature_schema import COLUMNAS_MODELO, FEATURES_HUELLA_802_3

extractor = ExtractorHuellaDigital()
print("Extractor de huella IEEE 802.3 instanciado correctamente desde src/fingerprint.py")
print(f"Características de enlace definidas: {FEATURES_HUELLA_802_3}")


## Fase 3 — Preparación del conjunto de datos

Como conjunto de datos de referencia se propone **CICIDS2017/2018** (Sharafaldin et al., 2018), por su cobertura de
tráfico legítimo y malicioso etiquetado (DDoS, port scan, brute force, entre otros) y su uso extendido en la
literatura de IDS basados en aprendizaje automático (Ahmad et al., 2021; Pinto et al., 2023).

**Nota de trazabilidad:** mientras se gestiona el acceso y la descarga del dataset completo (varios GB, requiere
almacenamiento externo), esta sección genera un conjunto de datos sintético con la misma estructura de
características (huella digital + variables de tráfico), para dejar el pipeline de modelado completamente
funcional y listo para sustituir por los datos reales sin cambiar el resto del cuaderno.


In [ ]:
# 3. Conjunto de datos — carga real o generación sintética con contrato unificado
from src.data import cargar_dataset, RUTA_DATASET_DEFECTO

df = cargar_dataset(RUTA_DATASET_DEFECTO, n_sintetico=5000)
print(f"Dataset cargado: {df.shape[0]} muestras, {df.shape[1]} columnas")
print("Distribución de clases:")
print(df["etiqueta"].value_counts(normalize=True).round(3))
display(df.head())


In [ ]:
# Partición estratificada y escalado con src/preprocess.py
from src.preprocess import dividir, ajustar_escalador, preprocesar

X_train, X_test, y_train, y_test = dividir(df, test_size=0.25)
scaler = ajustar_escalador(X_train)
X_train_esc = preprocesar(X_train, scaler)
X_test_esc = preprocesar(X_test, scaler)

print(f"Entrenamiento: {X_train.shape[0]} muestras | Prueba: {X_test.shape[0]} muestras")


## Fase 4 — Modelado: Random Forest vs. XGBoost

Se entrenan los dos modelos definidos en los objetivos específicos del proyecto para comparar su desempeño
en la clasificación del tráfico como legítimo o malicioso.


In [ ]:
# 4. Entrenamiento de modelos con src/models.py
from src.models import entrenar_random_forest, entrenar_xgboost

print("Entrenando Random Forest...")
modelo_rf = entrenar_random_forest(X_train_esc, y_train)

print("Entrenando XGBoost...")
modelo_xgb = entrenar_xgboost(X_train_esc, y_train)
print("Entrenamiento completado.")


## Fase 5 — Evaluación comparativa

Métricas: exactitud (accuracy), precisión, sensibilidad (recall), F1-score y tasa de falsos positivos (FPR),
conforme a lo definido en el tercer objetivo específico del proyecto.


In [ ]:
# 5. Evaluación comparativa importando evaluar() de src/models.py
from src.models import evaluar

res_rf = evaluar(modelo_rf, X_test_esc, y_test, "Random Forest")
res_xgb = evaluar(modelo_xgb, X_test_esc, y_test, "XGBoost")

resultados = pd.DataFrame([res_rf, res_xgb])
display(resultados.set_index("modelo").round(4))


In [ ]:
# Visualización comparativa
resultados_plot = resultados.set_index('modelo')[['accuracy', 'precision', 'recall', 'f1_score']]
resultados_plot.plot(kind='bar', figsize=(8, 5), rot=0)
plt.title('Comparación de desempeño: Random Forest vs. XGBoost')
plt.ylabel('Valor de la métrica')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Fase 6 — Simulación de ataque a la huella digital y evaluación de la defensa

Se simula un ataque de suplantación (spoofing) dirigido a alterar la huella digital de un dispositivo
(p. ej. modificando la MAC de origen o el patrón de tramas), y se evalúa si el modelo entrenado logra
seguir clasificando correctamente el tráfico como malicioso — es decir, la capacidad de defensa del prototipo.


In [ ]:
# 6. Simulación de ataque dirigido a la huella digital importando de src/
from src.attack_simulator import simular_ataque_spoofing

idx_legitimo = y_test[y_test == 0].index
X_legitimo_test = X_test.loc[idx_legitimo]
X_ataque = simular_ataque_spoofing(X_legitimo_test)
X_ataque_esc = preprocesar(X_ataque, scaler)
y_ataque_esperado = pd.Series(1, index=X_ataque.index)

deteccion = pd.DataFrame([
    evaluar(modelo_rf, X_ataque_esc, y_ataque_esperado, "Random Forest (bajo ataque)"),
    evaluar(modelo_xgb, X_ataque_esc, y_ataque_esperado, "XGBoost (bajo ataque)"),
])
display(deteccion.set_index("modelo")[["recall"]].rename(columns={"recall": "tasa_deteccion"}).round(4))


## Fase 7 — Diagnóstico, mitigación y pronóstico

- **Diagnóstico:** completar con el análisis comparativo de la Fase 5 una vez sustituido el dataset sintético
  por el dataset real (CICIDS2017/2018) y por la captura propia de huella digital.
- **Mitigación:** definir reglas de respuesta ante tráfico clasificado como anómalo (p. ej. cuarentena del
  dispositivo, alerta, bloqueo temporal por MAC/OUI).
- **Pronóstico:** con los resultados de la Fase 6, proyectar la tendencia de comportamiento anómalo en la red
  (por ejemplo, mediante una serie de tiempo de la tasa de detección de ataques a lo largo de varias simulaciones).

**Pendientes de trazabilidad (próximos cortes del curso):**
1. Reemplazar el dataset sintético por CICIDS2017/2018 real.
2. Ejecutar la captura de huella digital en el laboratorio (Fase 2) e integrarla como variable adicional.
3. Ajuste de hiperparámetros (`GridSearchCV` / `RandomizedSearchCV`) para ambos modelos.
4. Documentar el diagnóstico, la mitigación y el pronóstico con los resultados reales para el entregable final.
